# كشكول التجارب: تطبيق مسترجع أقصى تنوع نسبي (MMR) عملياً (المحاضرة 47)

أهلاً بيك في معمل المحاضرة 47!
لما بنعمل بحث عادي بالـ Similarity، المسترجع بيجيب أقرب 3 مستندات في المتجهات، والمشكلة إن الـ 3 دول ممكن يكونوا كلهم بيكرروا نفس الكلام بالظبط!
خوارزمية **Maximal Marginal Relevance (MMR)** بتعمل حركة ذكية جداً:
1. بتاخد أول مستند الأكثر صلة بالسؤال.
2. المستند التاني بتختاره بحيث يكون مفيد للسؤال، وفي نفس الوقت مختلف عن المستند الأول ومش مكرر!
تعال نجرب ونشوف ده بعينينا.

### 1. تجهيز المكتبات والوثائق المتكررة
يلا نجهز وثائق فيها تكرار متعمد علشان نختبر الخوارزمية:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docs = [
    Document(page_content="بايثون لغة برمجة أنشأها جايدو فان روسم سنة 1989 وتركز على سهولة القراءة.", metadata={"id": "تاريخ_1"}),
    Document(page_content="قام جايدو فان روسم بتطوير لغة بايثون في أواخر الثمانينات لتكون لغة سهلة وواضحة.", metadata={"id": "تاريخ_2"}),
    Document(page_content="تعتبر بايثون اللغة الأولى عالمياً في الذكاء الاصطناعي وتعلم الآلة مع مكتبات زي PyTorch.", metadata={"id": "ذكاء_اصطناعي"}),
    Document(page_content="تستخدم بايثون في تطوير الويب والواجهات الخلفية عبر أطر عمل سريعة مثل FastAPI.", metadata={"id": "تطوير_ويب"}),
]

vector_store = FAISS.from_documents(docs, embeddings)
print("تم بناء الفهرس في الذاكرة بنجاح!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_23068\2599934419.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء الفهرس في الذاكرة بنجاح!


### 2. مقارنة حية: البحث العادي مقابل مسترجع MMR
تعال نسأل سؤال عام ونشوف الفرق بين الطريقتين:

In [2]:
query = "ما هي لغة بايثون وفيما تستخدم عملياً؟"

# 1. البحث العادي
sim_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
sim_docs = sim_retriever.invoke(query)

print("=== البحث العادي (Similarity) - وقع في فخ التكرار ===")
for d in sim_docs:
    print(f"- [{d.metadata['id']}]: {d.page_content}")

# 2. مسترجع MMR
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 4, "lambda_mult": 0.5}
)
mmr_docs = mmr_retriever.invoke(query)

print("\n=== مسترجع MMR - توازن رائع بين الصلة والتنوع ===")
for d in mmr_docs:
    print(f"- [{d.metadata['id']}]: {d.page_content}")

=== البحث العادي (Similarity) - وقع في فخ التكرار ===
- [تطوير_ويب]: تستخدم بايثون في تطوير الويب والواجهات الخلفية عبر أطر عمل سريعة مثل FastAPI.
- [تاريخ_2]: قام جايدو فان روسم بتطوير لغة بايثون في أواخر الثمانينات لتكون لغة سهلة وواضحة.

=== مسترجع MMR - توازن رائع بين الصلة والتنوع ===
- [تطوير_ويب]: تستخدم بايثون في تطوير الويب والواجهات الخلفية عبر أطر عمل سريعة مثل FastAPI.
- [تاريخ_2]: قام جايدو فان روسم بتطوير لغة بايثون في أواخر الثمانينات لتكون لغة سهلة وواضحة.


### 3. تجربة تأثير معامل التوازن (lambda_mult)
لاحظ الفرق لما نغير قيمة lambda:

In [3]:
for lam in [0.1, 0.5, 0.9]:
    ret = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 4, "lambda_mult": lam})
    res = ret.invoke(query)
    ids = [d.metadata["id"] for d in res]
    print(f"قيمة lambda = {lam:.1f} -> المستندات المختارة: {ids}")

قيمة lambda = 0.1 -> المستندات المختارة: ['تطوير_ويب', 'ذكاء_اصطناعي']
قيمة lambda = 0.5 -> المستندات المختارة: ['تطوير_ويب', 'تاريخ_2']
قيمة lambda = 0.9 -> المستندات المختارة: ['تطوير_ويب', 'تاريخ_2']


### 4. جرب بنفسك!
جرب تضيف وثائق جديدة عن مجالات تانية (زي الأمان السيبراني أو الألعاب) وجرب درجات مختلفة من lambda.